In [2]:
# %% CELL 0 -- mount the offline package bundle + face-alignment hub cache
# (same bundle used by the training notebook -- face-alignment still needs
# to run here, since landmark detection is part of the real inference path)
import os, shutil, sys, zipfile

OFFLINE_PACKAGES_SRC = "/kaggle/input/datasets/ruwadnaswan/package-df-msib/offline_packages"      # EDIT
FACE_ALIGNMENT_HUB_SRC = "/kaggle/input/datasets/ruwadnaswan/package-df-msib/face_alignment_hub"   # EDIT

def _materialize(source, dest_dir):
    if os.path.isdir(source):
        return source
    if os.path.isfile(source) and source.lower().endswith(".zip"):
        if not (os.path.isdir(dest_dir) and os.listdir(dest_dir)):
            os.makedirs(dest_dir, exist_ok=True)
            with zipfile.ZipFile(source, "r") as z:
                z.extractall(dest_dir)
        return dest_dir
    raise FileNotFoundError(f"{source} is neither an existing directory nor a .zip file")

SITE_PACKAGES_DIR = _materialize(OFFLINE_PACKAGES_SRC, "/kaggle/working/offline_site_packages")
sys.path.insert(0, SITE_PACKAGES_DIR)

HUB_DIR = "/kaggle/working/hub"
if not (os.path.isdir(HUB_DIR) and os.listdir(HUB_DIR)):
    if os.path.isdir(FACE_ALIGNMENT_HUB_SRC):
        shutil.copytree(FACE_ALIGNMENT_HUB_SRC, HUB_DIR, dirs_exist_ok=True)
    else:
        _materialize(FACE_ALIGNMENT_HUB_SRC, HUB_DIR)

import torch  # noqa: E402
torch.hub.set_dir(HUB_DIR)
import face_alignment  # noqa: E402
print("face_alignment OK, version:", getattr(face_alignment, "__version__", "unknown"))
print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "bf16 supported:", torch.cuda.is_bf16_supported())

# Perf knobs: fixed input shapes every call (B=1, T=8, 224x224) -> let cudnn
# autotune once during warmup and reuse the winning algorithm forever after.
torch.backends.cudnn.benchmark = True


# %% CELL 1 -- imports & config (inference-only subset of the training CFG --
# same field names/values so a checkpoint trained with the training notebook's
# defaults loads with matching shapes)
import math, time, statistics
import numpy as np
import cv2
import torch.nn as nn
import torch.nn.functional as F
from collections import deque
from typing import List, Tuple

torch.set_float32_matmul_precision("high")
NEG_INF = -1e4

THRESHOLD = 0.1


class CFG:
    img_size = 224
    patch_size = 16
    num_frames = 8
    num_landmarks = 68
    max_hop = 2
    num_freq_bands = 6

    vis_dim = 384
    vis_depth = 12
    vis_heads = 6
    graph_dim = 192
    graph_depth = 12
    graph_heads = 6
    mlp_ratio = 4.0
    num_classes = 2

    guide_bias_clip = 8.0

    amp = True
    amp_dtype = "bf16"

    celebdf_crop_margin = 0.4   # used for the video-preprocessing face crop

    # torch.compile toggle -- flip off if you need to A/B against eager mode,
    # or if you hit a compile error you don't want to debug right now.
    use_compile = True
    compile_mode = "reduce-overhead"   # good fit: fixed shapes, CUDA graphs

    device = "cuda" if torch.cuda.is_available() else "cpu"


def resolve_amp_dtype(cfg: "CFG"):
    if cfg.amp_dtype == "bf16" and torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    if cfg.amp_dtype == "bf16":
        print("[amp] bf16 requested but unsupported on this GPU -- falling back to fp16")
    return torch.float16


# %% CELL 2 -- landmark graph utilities (identical to the training notebook's CELL 3)
def _ibug68_edges() -> List[Tuple[int, int]]:
    edges = []
    edges += [(i, i + 1) for i in range(0, 16)]
    edges += [(i, i + 1) for i in range(17, 21)]
    edges += [(i, i + 1) for i in range(22, 26)]
    edges += [(i, i + 1) for i in range(27, 30)]
    edges += [(i, i + 1) for i in range(31, 35)]
    edges += [(27, 31), (30, 33)]
    edges += [(i, i + 1) for i in range(36, 41)] + [(41, 36)]
    edges += [(i, i + 1) for i in range(42, 47)] + [(47, 42)]
    edges += [(i, i + 1) for i in range(48, 59)] + [(59, 48)]
    edges += [(i, i + 1) for i in range(60, 67)] + [(67, 60)]
    edges += [(19, 37), (24, 44), (33, 51)]
    return edges


EDGES_68 = _ibug68_edges()
NUM_LANDMARKS = 68


def build_adjacency(num_nodes: int = NUM_LANDMARKS, edges=EDGES_68) -> np.ndarray:
    adj = np.zeros((num_nodes, num_nodes), dtype=bool)
    for i, j in edges:
        adj[i, j] = True
        adj[j, i] = True
    return adj


def build_hyper_adjacency(num_landmarks: int = NUM_LANDMARKS) -> np.ndarray:
    base = build_adjacency(num_landmarks)
    adj = np.zeros((num_landmarks + 1, num_landmarks + 1), dtype=bool)
    adj[1:, 1:] = base
    adj[0, 1:] = True
    adj[1:, 0] = True
    return adj


def all_pairs_hops(adj: np.ndarray, max_hop: int) -> np.ndarray:
    n = adj.shape[0]
    hop = np.full((n, n), -1, dtype=np.int64)
    for src in range(n):
        hop[src, src] = 0
        q = deque([src]); visited = {src}
        while q:
            u = q.popleft()
            if hop[src, u] >= max_hop:
                continue
            for v in np.nonzero(adj[u])[0]:
                if v not in visited:
                    visited.add(v)
                    hop[src, v] = hop[src, u] + 1
                    q.append(v)
    return hop


def node_degree(adj: np.ndarray) -> np.ndarray:
    return adj.sum(axis=1).astype(np.int64)


# %% CELL 3 -- shared embedding modules (identical to training notebook's CELL 7)
class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=384):
        super().__init__()
        self.grid = img_size // patch_size
        self.num_patches = self.grid * self.grid
        self.patch_size = patch_size
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.reshape(B * T, C, H, W)
        x = self.proj(x).flatten(2).transpose(1, 2)
        D = x.shape[-1]
        return x.reshape(B, T, -1, D)


def sine_cosine_pe(coords, num_freq_bands):
    freqs = (2.0 ** torch.arange(num_freq_bands, device=coords.device, dtype=coords.dtype)) * math.pi
    args = coords.unsqueeze(-1) * freqs
    enc = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
    return enc.flatten(-2)


class NodeEmbed(nn.Module):
    def __init__(self, embed_dim, num_freq_bands, num_nodes=NUM_LANDMARKS, max_degree=8):
        super().__init__()
        in_dim = 3 * 2 * num_freq_bands
        self.proj = nn.Linear(in_dim, embed_dim)
        degree = torch.from_numpy(node_degree(build_adjacency(num_nodes))).long()
        self.register_buffer("degree", degree)
        self.degree_embed = nn.Embedding(max_degree + 1, embed_dim)
        self.num_freq_bands = num_freq_bands

    def forward(self, landmarks_norm):
        pe = sine_cosine_pe(landmarks_norm, self.num_freq_bands)
        h = self.proj(pe)
        deg = self.degree.clamp(max=self.degree_embed.num_embeddings - 1)
        return h + self.degree_embed(deg)[None, None, :, :]


# %% CELL 4 -- spatial attention (vision stream)
# CHANGED: uses F.scaled_dot_product_attention (fused flash/mem-efficient
# kernel on GPU) instead of manual matmul+softmax+matmul. Safe here because
# the caller never uses the post-softmax attention weights -- only `out`.
class RelPosBias(nn.Module):
    def __init__(self, grid, num_heads):
        super().__init__()
        self.grid = grid
        num_rel = (2 * grid - 1) * (2 * grid - 1)
        self.table = nn.Parameter(torch.zeros(num_rel, num_heads))
        coords = torch.stack(torch.meshgrid(torch.arange(grid), torch.arange(grid), indexing="ij"), dim=-1).reshape(-1, 2)
        rel = coords[:, None, :] - coords[None, :, :] + (grid - 1)
        idx = rel[..., 0] * (2 * grid - 1) + rel[..., 1]
        self.register_buffer("index", idx)
        nn.init.trunc_normal_(self.table, std=0.02)

    def forward(self, has_cls: bool):
        L = self.grid * self.grid
        bias = self.table[self.index.reshape(-1)].reshape(L, L, -1).permute(2, 0, 1)
        if has_cls:
            H = bias.shape[0]
            full = bias.new_zeros(H, L + 1, L + 1)
            full[:, 1:, 1:] = bias
            bias = full
        return bias


class SpatialAttention(nn.Module):
    def __init__(self, dim, num_heads, grid, has_cls=True):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        self.rel_pos = RelPosBias(grid, num_heads)
        self.has_cls = has_cls

    def forward(self, x):
        B, T, L, D = x.shape
        qkv = self.qkv(x).reshape(B, T, L, 3, self.num_heads, self.head_dim).permute(3, 0, 1, 4, 2, 5)
        q, k, v = qkv[0], qkv[1], qkv[2]
        bias = self.rel_pos(self.has_cls)[None, None].to(q.dtype)  # (1,1,H,L,L) broadcasts over (B,T,H,L,L)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=bias, scale=self.scale)
        out = out.transpose(2, 3).reshape(B, T, L, D)
        return self.proj(out)


# %% CELL 5 -- topology-aware spatial attention (graph stream)
# CHANGED: also switched to SDPA -- the caller (GraphSTB) never uses these
# post-softmax attention weights either, only the KTA's raw_logits matter
# downstream (for guide_bias), and those come from a different sub-module.
class TopoSpatialAttention(nn.Module):
    def __init__(self, dim, num_heads, num_landmarks=NUM_LANDMARKS, max_hop=2):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

        adj = build_hyper_adjacency(num_landmarks)
        hop = all_pairs_hops(adj, max_hop)
        self.register_buffer("hop", torch.from_numpy(hop).long())
        self.hop_embed = nn.Parameter(torch.zeros(max_hop + 1, num_heads))
        nn.init.trunc_normal_(self.hop_embed, std=0.02)

    def _bias(self, device, dtype):
        N = self.hop.shape[0]
        valid = self.hop >= 0
        idx = self.hop.clamp(min=0)
        gathered = self.hop_embed[idx.reshape(-1)].reshape(N, N, self.num_heads)
        bias = torch.where(valid.unsqueeze(-1), gathered, torch.full_like(gathered, NEG_INF))
        return bias.permute(2, 0, 1).to(dtype)  # (H, N, N)

    def forward(self, x):
        B, T, N, D = x.shape
        qkv = self.qkv(x).reshape(B, T, N, 3, self.num_heads, self.head_dim).permute(3, 0, 1, 4, 2, 5)
        q, k, v = qkv[0], qkv[1], qkv[2]
        bias = self._bias(x.device, q.dtype)[None, None]  # (1,1,H,N,N)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=bias, scale=self.scale)
        out = out.transpose(2, 3).reshape(B, T, N, D)
        return self.proj(out)


# %% CELL 6 -- Kronecker Temporal Attention (KTA)
# CHANGED: added `need_raw_logits`. The graph stream's KTA output feeds
# extract_landmark_block -> guide_bias, so it MUST return real pre-softmax
# logits (manual matmul path, unchanged numerically). The vision stream's
# KTA output is only ever used as `x = x + ta_out` -- its raw_logits are
# discarded by the caller -- so it's safe to route through fused SDPA
# instead, passing (kronecker_mask + guide_bias) as one additive attn_mask.
import functools

@functools.lru_cache(maxsize=8)
def kronecker_temporal_mask(T, L, device, dtype):
    I_t = torch.eye(T, device=device)
    J_m = torch.ones(L, L, device=device)
    I_m = torch.eye(L, device=device)
    M = torch.kron(I_t, (J_m - I_m))
    return torch.where(M > 0, torch.full_like(M, NEG_INF), torch.zeros_like(M)).to(dtype)


class KroneckerTemporalAttention(nn.Module):
    def __init__(self, dim, num_heads, need_raw_logits=True):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        self.need_raw_logits = need_raw_logits

    def forward(self, x, guide_bias=None):
        B, T, L, D = x.shape
        x_flat = x.reshape(B, T * L, D)
        qkv = self.qkv(x_flat).reshape(B, T * L, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        mask = kronecker_temporal_mask(T, L, x.device, q.dtype)[None, None]  # (1,1,TL,TL)

        if self.need_raw_logits:
            raw_logits = (q @ k.transpose(-2, -1)) * self.scale
            logits = raw_logits + mask
            if guide_bias is not None:
                logits = logits + guide_bias[:, None]
            attn = logits.softmax(dim=-1)
            out = attn @ v
        else:
            attn_mask = mask if guide_bias is None else (mask + guide_bias[:, None])
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, scale=self.scale)
            raw_logits = None

        out = out.transpose(1, 2).reshape(B, T * L, D)
        out = self.proj(out).reshape(B, T, L, D)
        return out, raw_logits


def extract_landmark_block(raw_logits, T, N):
    B, H = raw_logits.shape[0], raw_logits.shape[1]
    Np1 = N + 1
    r = raw_logits.view(B, H, T, Np1, T, Np1)
    r = r[:, :, :, 1:, :, 1:]
    return r.reshape(B, H, T * N, T * N)


# %% CELL 6b -- scatter-index caching (NEW)
# The old scatter_graph_bias_to_vision() rebuilt t_idx / node_to_patch /
# row_full / idx_i / idx_j / batch_offset / flat_idx from scratch on EVERY
# call -- and it's called once per transformer layer (12x per forward),
# even though none of that indexing depends on the layer, only on
# `patch_idx`, which is fixed for the whole forward pass. Split it: compute
# indices once per forward, then just scatter_add the values per layer.
def build_scatter_indices(patch_idx, T, N, L, B, device):
    TL, TN = T * L, T * N
    t_idx = torch.arange(T, device=device).repeat_interleave(N)
    node_to_patch = patch_idx.reshape(B, TN)
    row_full = t_idx.unsqueeze(0) * L + node_to_patch

    idx_i = row_full.unsqueeze(2).expand(B, TN, TN).reshape(B, -1)
    idx_j = row_full.unsqueeze(1).expand(B, TN, TN).reshape(B, -1)
    batch_offset = (torch.arange(B, device=device) * TL * TL).unsqueeze(1)
    flat_idx = (idx_i * TL + idx_j + batch_offset).reshape(-1)

    out_cnt = torch.zeros(B * TL * TL, device=device, dtype=torch.float32)
    out_cnt.scatter_add_(0, flat_idx, torch.ones_like(flat_idx, dtype=torch.float32))
    return flat_idx, out_cnt.clamp(min=1.0)


def scatter_graph_bias_to_vision(graph_raw_logits_land, flat_idx, out_cnt_clamped, B, TL, bias_clip=8.0):
    out_dtype = graph_raw_logits_land.dtype
    graph_bias = graph_raw_logits_land.float().mean(dim=1)  # (B, TN, TN)

    out_sum = torch.zeros(B * TL * TL, device=graph_bias.device, dtype=torch.float32)
    out_sum.scatter_add_(0, flat_idx, graph_bias.reshape(-1))

    out = (out_sum / out_cnt_clamped).reshape(B, TL, TL)
    out = out.clamp(min=-bias_clip, max=bias_clip)
    return out.to(out_dtype)


# %% CELL 7 -- spatiotemporal blocks
# CHANGED: SpatialAttention / TopoSpatialAttention now return a single
# tensor (no unused attn weights) -- call sites simplified accordingly.
# VisionSTB's KTA runs with need_raw_logits=False (SDPA path); GraphSTB's
# KTA keeps need_raw_logits=True (manual path, feeds guide_bias).
class Mlp(nn.Module):
    def __init__(self, dim, ratio=4.0, drop=0.0):
        super().__init__()
        hidden = int(dim * ratio)
        self.fc1 = nn.Linear(dim, hidden)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden, dim)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        return self.fc2(self.drop(self.act(self.fc1(x))))


class VisionSTB(nn.Module):
    def __init__(self, dim, num_heads, grid, mlp_ratio=4.0, has_cls=True):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.spatial = SpatialAttention(dim, num_heads, grid, has_cls=has_cls)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp1 = Mlp(dim, mlp_ratio)
        self.norm3 = nn.LayerNorm(dim)
        self.temporal = KroneckerTemporalAttention(dim, num_heads, need_raw_logits=False)
        self.norm4 = nn.LayerNorm(dim)
        self.mlp2 = Mlp(dim, mlp_ratio)

    def forward(self, x, guide_bias=None):
        x = x + self.spatial(self.norm1(x))
        x = x + self.mlp1(self.norm2(x))
        ta_out, _ = self.temporal(self.norm3(x), guide_bias=guide_bias)
        x = x + ta_out
        x = x + self.mlp2(self.norm4(x))
        return x


class GraphSTB(nn.Module):
    def __init__(self, dim, num_heads, num_landmarks, max_hop, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.spatial = TopoSpatialAttention(dim, num_heads, num_landmarks, max_hop)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp1 = Mlp(dim, mlp_ratio)
        self.norm3 = nn.LayerNorm(dim)
        self.temporal = KroneckerTemporalAttention(dim, num_heads, need_raw_logits=True)
        self.norm4 = nn.LayerNorm(dim)
        self.mlp2 = Mlp(dim, mlp_ratio)

    def forward(self, x):
        x = x + self.spatial(self.norm1(x))
        x = x + self.mlp1(self.norm2(x))
        ta_out, raw_logits = self.temporal(self.norm3(x))
        x = x + ta_out
        x = x + self.mlp2(self.norm4(x))
        return x, raw_logits


# %% CELL 8 -- full G2V2former model
# CHANGED: forward() now builds the scatter indices ONCE (build_scatter_indices)
# before the layer loop, instead of rebuilding them inside every call to
# scatter_graph_bias_to_vision (12x redundant work removed).
class G2V2former(nn.Module):
    def __init__(self, cfg: CFG):
        super().__init__()
        self.T = cfg.num_frames
        self.N = cfg.num_landmarks
        self.patch_size = cfg.patch_size
        self.img_size = cfg.img_size
        self.guide_bias_clip = cfg.guide_bias_clip

        self.patch_embed = PatchEmbed(cfg.img_size, cfg.patch_size, 3, cfg.vis_dim)
        self.grid = self.patch_embed.grid
        self.L = self.patch_embed.num_patches
        self.vis_pos_embed = nn.Parameter(torch.zeros(1, 1, self.L, cfg.vis_dim))
        self.vis_time_embed = nn.Parameter(torch.zeros(1, cfg.num_frames, 1, cfg.vis_dim))
        self.vis_cls_token = nn.Parameter(torch.zeros(1, 1, 1, cfg.vis_dim))
        self.vis_blocks = nn.ModuleList([
            VisionSTB(cfg.vis_dim, cfg.vis_heads, self.grid, cfg.mlp_ratio, has_cls=True)
            for _ in range(cfg.vis_depth)
        ])
        self.vis_norm = nn.LayerNorm(cfg.vis_dim)

        self.node_embed = NodeEmbed(cfg.graph_dim, cfg.num_freq_bands, cfg.num_landmarks)
        self.hyper_node = nn.Parameter(torch.zeros(1, 1, 1, cfg.graph_dim))
        self.graph_time_embed = nn.Parameter(torch.zeros(1, cfg.num_frames, 1, cfg.graph_dim))
        self.graph_blocks = nn.ModuleList([
            GraphSTB(cfg.graph_dim, cfg.graph_heads, cfg.num_landmarks, cfg.max_hop, cfg.mlp_ratio)
            for _ in range(cfg.graph_depth)
        ])
        self.graph_norm = nn.LayerNorm(cfg.graph_dim)

        self.head = nn.Linear(cfg.vis_dim + cfg.graph_dim, cfg.num_classes)

        nn.init.trunc_normal_(self.vis_pos_embed, std=0.02)
        nn.init.trunc_normal_(self.vis_time_embed, std=0.02)
        nn.init.trunc_normal_(self.vis_cls_token, std=0.02)
        nn.init.trunc_normal_(self.hyper_node, std=0.02)
        nn.init.trunc_normal_(self.graph_time_embed, std=0.02)

    def forward(self, frames, landmarks_norm, landmarks_px):
        B, T = frames.shape[0], frames.shape[1]

        x = self.patch_embed(frames)
        x = x + self.vis_pos_embed + self.vis_time_embed
        cls_tok = self.vis_cls_token.expand(B, T, -1, -1)
        x = torch.cat([cls_tok, x], dim=2)
        L_full = x.shape[2]

        g = self.node_embed(landmarks_norm) + self.graph_time_embed
        hnode = self.hyper_node.expand(B, T, -1, -1)
        g = torch.cat([hnode, g], dim=2)

        with torch.no_grad():
            grid = self.grid
            px = landmarks_px.clamp(min=0, max=self.img_size - 1)
            col = (px[..., 0] // self.patch_size).long().clamp(max=grid - 1)
            row = (px[..., 1] // self.patch_size).long().clamp(max=grid - 1)
            patch_idx = (row * grid + col) + 1

        # NEW: index prep done once per forward, not once per layer.
        TL = T * L_full
        flat_idx, out_cnt_clamped = build_scatter_indices(patch_idx, T, self.N, L_full, B, x.device)

        guide_bias = None
        for vblock, gblock in zip(self.vis_blocks, self.graph_blocks):
            g, g_raw_logits = gblock(g)
            g_land_logits = extract_landmark_block(g_raw_logits, T, self.N)
            guide_bias = scatter_graph_bias_to_vision(
                g_land_logits, flat_idx, out_cnt_clamped, B, TL, bias_clip=self.guide_bias_clip
            )
            x = vblock(x, guide_bias=guide_bias)

        x = self.vis_norm(x)
        g = self.graph_norm(g)

        vis_cls_seq = x[:, :, 0, :]
        vis_cls_out = vis_cls_seq.mean(dim=1)
        graph_hyper_out = g[:, :, 0, :].mean(dim=1)

        feat = torch.cat([vis_cls_out, graph_hyper_out], dim=-1)
        logits = self.head(feat)
        return logits, vis_cls_out, graph_hyper_out, vis_cls_seq


# %% CELL 9 -- load a checkpoint saved by the training notebook
# The training notebook wrote checkpoints under its own /kaggle/working/ckpts,
# which does NOT persist to a separate notebook/session. Publish that folder
# (or just the .pt files you need) as a Kaggle Dataset and mount it here.
CKPT_INPUT_DIR = "/kaggle/input/notebooks/sleepytmzd/lets-see-2/ckpts"   # EDIT: wherever you published notebook 1's ckpts


def load_model_from_checkpoint(cfg: "CFG", ckpt_path: str, compile_model: bool = None) -> nn.Module:
    """Loads weights into an eager G2V2former, then optionally wraps it with
    torch.compile. `compile_model` defaults to cfg.use_compile if not given.

    NOTE: the returned module, if compiled, is a new object (torch.compile
    returns an OptimizedModule wrapper) but state_dict()/parameters() still
    work normally for anything that needs the underlying weights."""
    model = G2V2former(cfg).to(cfg.device)
    state = torch.load(ckpt_path, map_location=cfg.device, weights_only=True)
    model.load_state_dict(state)
    model.eval()

    do_compile = cfg.use_compile if compile_model is None else compile_model
    if do_compile and cfg.device == "cuda":
        model = torch.compile(model, mode=cfg.compile_mode)
        print(f"[compile] wrapped model with torch.compile(mode='{cfg.compile_mode}') "
              f"-- first call(s) will be slow while it traces/autotunes, this is expected.")
    elif do_compile and cfg.device != "cuda":
        print("[compile] skipped -- torch.compile(mode='reduce-overhead') requires CUDA")
    return model


# %% CELL 10 -- face-crop helper for video frames (identical to training notebook's CELL 6)
def _crop_box_and_landmarks(preds, h, w, margin, img_size):
    if not preds:
        side = int(min(h, w) * 0.8)
        y0, x0 = (h - side) // 2, (w - side) // 2
        x1, y1 = x0 + side, y0 + side
        lmk = np.zeros((NUM_LANDMARKS, 3), dtype=np.float32)
        return (x0, y0, x1, y1), lmk

    p = preds[0].astype(np.float32)
    xy = p[:, :2]
    x_min, y_min = xy.min(axis=0)
    x_max, y_max = xy.max(axis=0)
    bw, bh = max(x_max - x_min, 1.0), max(y_max - y_min, 1.0)
    cx, cy = (x_min + x_max) / 2, (y_min + y_max) / 2
    side = max(bw, bh) * (1.0 + margin)
    x0, x1 = int(max(0, cx - side / 2)), int(min(w, cx + side / 2))
    y0, y1 = int(max(0, cy - side / 2)), int(min(h, cy + side / 2))
    if x1 <= x0 or y1 <= y0:
        x0, y0, x1, y1 = 0, 0, w, h

    sx, sy = img_size / (x1 - x0), img_size / (y1 - y0)
    lmk = p.copy()
    lmk[:, 0] = (lmk[:, 0] - x0) * sx
    lmk[:, 1] = (lmk[:, 1] - y0) * sy
    if not np.isfinite(lmk).all():
        lmk = np.zeros((NUM_LANDMARKS, 3), dtype=np.float32)
    return (x0, y0, x1, y1), lmk


# %% CELL 11 -- preprocessing: raw file on disk -> model-ready tensors (batch size 1)
def _preprocess_still_image(img_bgr, fa_model, cfg: "CFG"):
    """Image pipeline: no face-box crop, direct resize (matches face-cropped
    still datasets like LCC-FASD / Asian-Fakes). A still image is a 'clip of
    length 1' -- replicated across cfg.num_frames temporal slots."""
    h0, w0 = img_bgr.shape[:2]
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    try:
        preds = fa_model.get_landmarks(img_rgb)
    except Exception:
        preds = None
    lmk = preds[0].astype(np.float32) if preds else np.zeros((NUM_LANDMARKS, 3), dtype=np.float32)
    if not np.isfinite(lmk).all():
        lmk = np.zeros((NUM_LANDMARKS, 3), dtype=np.float32)

    img_resized = cv2.resize(img_rgb, (cfg.img_size, cfg.img_size))
    sx, sy = cfg.img_size / w0, cfg.img_size / h0
    lmk = lmk.copy()
    lmk[:, 0] *= sx
    lmk[:, 1] *= sy

    img_t = torch.from_numpy(img_resized).permute(2, 0, 1).float() / 255.0
    img_t = (img_t - 0.5) / 0.5
    frames = img_t.unsqueeze(0).repeat(cfg.num_frames, 1, 1, 1)

    lmks_px = torch.from_numpy(lmk[:, :2]).unsqueeze(0).repeat(cfg.num_frames, 1, 1)
    lmks_norm = lmks_px.clone()
    lmks_norm[..., 0] /= cfg.img_size
    lmks_norm[..., 1] /= cfg.img_size
    z = torch.from_numpy(lmk[:, 2]).unsqueeze(0).repeat(cfg.num_frames, 1)
    z_range = (z.max() - z.min()) + 1e-6
    z_norm = (z - z.min()) / z_range
    lmks_norm = torch.cat([lmks_norm, z_norm.unsqueeze(-1)], dim=-1)

    return frames.unsqueeze(0), lmks_norm.unsqueeze(0), lmks_px.unsqueeze(0)


def _preprocess_video(video_path, fa_model, cfg: "CFG"):
    """Video pipeline: decode cfg.num_frames sampled frames, face-landmark
    detect + crop + resize each (matches CelebDF-style raw video with
    background).

    CHANGED: landmark detection is now batched via fa_model.get_landmarks_from_batch
    when available (newer `face_alignment` releases expose this), instead of
    calling get_landmarks() once per frame in a Python loop. This amortizes
    the FAN model's kernel-launch/dispatch overhead across all num_frames
    frames in a single forward pass instead of paying it num_frames times.
    Falls back to the original per-frame loop if the batched API isn't
    present in your installed version."""
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        raise RuntimeError(f"Could not read frame count from {video_path}")
    T = cfg.num_frames
    target = set(np.linspace(0, total - 1, min(T, total)).round().astype(int).tolist())

    raw, i = {}, 0
    while len(raw) < len(target):
        ok = cap.grab()
        if not ok:
            break
        if i in target:
            ok, frame = cap.retrieve()
            if ok:
                raw[i] = frame
        i += 1
    cap.release()

    chosen = sorted(raw.keys())
    if not chosen:
        raise RuntimeError(f"No frames decoded from {video_path}")
    if len(chosen) < T:
        chosen = chosen + [chosen[-1]] * (T - len(chosen))
    chosen = chosen[:T]

    frames_rgb = [cv2.cvtColor(raw[idx], cv2.COLOR_BGR2RGB) for idx in chosen]

    preds_per_frame = _batched_get_landmarks(fa_model, frames_rgb)

    imgs, lmks = [], []
    for frame_rgb, preds in zip(frames_rgb, preds_per_frame):
        h, w = frame_rgb.shape[:2]
        box, lmk = _crop_box_and_landmarks(preds, h, w, cfg.celebdf_crop_margin, cfg.img_size)
        x0, y0, x1, y1 = box
        crop = frame_rgb[y0:y1, x0:x1]
        if crop.size == 0:
            crop = frame_rgb
        crop = cv2.resize(crop, (cfg.img_size, cfg.img_size))
        imgs.append(crop)
        lmks.append(lmk)

    imgs = np.stack(imgs)
    lmks = np.stack(lmks).astype(np.float32)
    imgs_t = torch.from_numpy(imgs).permute(0, 3, 1, 2).float() / 255.0
    imgs_t = (imgs_t - 0.5) / 0.5
    lmks_px = torch.from_numpy(lmks[..., :2])
    lmks_norm = lmks_px.clone()
    lmks_norm[..., 0] /= cfg.img_size
    lmks_norm[..., 1] /= cfg.img_size
    z = torch.from_numpy(lmks[..., 2])
    z_range = (z.max() - z.min()) + 1e-6
    z_norm = (z - z.min()) / z_range
    lmks_norm = torch.cat([lmks_norm, z_norm.unsqueeze(-1)], dim=-1)

    return imgs_t.unsqueeze(0), lmks_norm.unsqueeze(0), lmks_px.unsqueeze(0)


def _batched_get_landmarks(fa_model, frames_rgb):
    """Try face_alignment's batched API first (get_landmarks_from_batch, present
    in newer versions and expects a stacked uint8/float tensor of shape
    (N,C,H,W) or a list of same-size arrays depending on version). Falls back
    to the original one-call-per-frame loop, so this is safe on older
    installs of `face_alignment` too -- just without the speedup."""
    if hasattr(fa_model, "get_landmarks_from_batch"):
        try:
            batch = torch.from_numpy(np.stack(frames_rgb)).permute(0, 3, 1, 2).contiguous()
            preds_batch = fa_model.get_landmarks_from_batch(batch.float())
            # normalize return shape across versions: list-of-list-of-face -> take first face per frame
            out = []
            for p in preds_batch:
                if p is None or len(p) == 0:
                    out.append(None)
                else:
                    first = np.asarray(p[0]) if not isinstance(p, np.ndarray) else p
                    out.append([first] if first.ndim == 2 else [first[0]])
            return out
        except Exception as e:
            print(f"[landmarks] batched API failed ({e}), falling back to per-frame loop")

    out = []
    for frame_rgb in frames_rgb:
        try:
            preds = fa_model.get_landmarks(frame_rgb)
        except Exception:
            preds = None
        out.append(preds)
    return out


# %% CELL 12 -- timing harness + benchmark functions
# CHANGED: torch.inference_mode() instead of torch.no_grad() (skips autograd
# version-counter bookkeeping too); amp_dtype resolved once outside the
# per-call closures instead of being recomputed on every single run.
def _summarize_latency(records, tag):
    def _stats(key):
        vals = sorted(r[key] for r in records)
        return {
            "mean_ms": statistics.mean(vals),
            "median_ms": statistics.median(vals),
            "std_ms": statistics.pstdev(vals) if len(vals) > 1 else 0.0,
            "min_ms": vals[0], "max_ms": vals[-1],
            "p95_ms": vals[int(0.95 * (len(vals) - 1))],
        }

    summary = {"tag": tag, "n_runs": len(records),
               "preprocess": _stats("preprocess_ms"), "model": _stats("model_ms"), "total": _stats("total_ms")}

    print(f"\n=== Latency: {tag}  (n={len(records)} runs, batch size 1) ===")
    for stage in ("preprocess", "model", "total"):
        s = summary[stage]
        print(f"  {stage:11s} mean={s['mean_ms']:7.2f}ms  median={s['median_ms']:7.2f}ms  "
              f"p95={s['p95_ms']:7.2f}ms  std={s['std_ms']:6.2f}ms  "
              f"min={s['min_ms']:7.2f}ms  max={s['max_ms']:7.2f}ms")
    print(f"  -> throughput: {1000.0 / summary['total']['mean_ms']:.2f} inferences/sec "
          f"(end-to-end, incl. face-landmark detection)")
    return summary


@torch.inference_mode()
def _run_forward(model, frames, lmks_norm, lmks_px, cfg, device, amp_dtype):
    amp_enabled = cfg.amp and device == "cuda"
    frames = frames.to(device, non_blocking=True)
    lmks_norm = lmks_norm.to(device, non_blocking=True)
    lmks_px = lmks_px.to(device, non_blocking=True)
    if device == "cuda":
        torch.cuda.synchronize()
    t1 = time.perf_counter()

    with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=amp_enabled):
        logits, *_ = model(frames, lmks_norm, lmks_px)
        probs = F.softmax(logits.float(), dim=-1)[:, 1]
    if device == "cuda":
        torch.cuda.synchronize()
    t2 = time.perf_counter()
    return probs.item(), t1, t2


def benchmark_image_latency(image_path, model, cfg: "CFG", fa_model, n_warmup=5, n_runs=50):
    device = cfg.device
    amp_dtype = resolve_amp_dtype(cfg)
    model.eval()
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise FileNotFoundError(image_path)

    def _run_once():
        t0 = time.perf_counter()
        frames, lmks_norm, lmks_px = _preprocess_still_image(img_bgr, fa_model, cfg)
        p_live, t1, t2 = _run_forward(model, frames, lmks_norm, lmks_px, cfg, device, amp_dtype)
        return {"preprocess_ms": (t1 - t0) * 1000, "model_ms": (t2 - t1) * 1000,
                "total_ms": (t2 - t0) * 1000, "p_live": p_live}

    for _ in range(n_warmup):
        _run_once()
    records = [_run_once() for _ in range(n_runs)]
    return _summarize_latency(records, tag=f"IMAGE ({os.path.basename(image_path)})")


def benchmark_video_latency(video_path, model, cfg: "CFG", fa_model, n_warmup=3, n_runs=20):
    device = cfg.device
    amp_dtype = resolve_amp_dtype(cfg)
    model.eval()

    def _run_once():
        t0 = time.perf_counter()
        frames, lmks_norm, lmks_px = _preprocess_video(video_path, fa_model, cfg)
        p_live, t1, t2 = _run_forward(model, frames, lmks_norm, lmks_px, cfg, device, amp_dtype)
        return {"preprocess_ms": (t1 - t0) * 1000, "model_ms": (t2 - t1) * 1000,
                "total_ms": (t2 - t0) * 1000, "p_live": p_live}

    for _ in range(n_warmup):
        _run_once()
    records = [_run_once() for _ in range(n_runs)]
    return _summarize_latency(records, tag=f"VIDEO ({os.path.basename(video_path)}, {cfg.num_frames} frames sampled)")


@torch.inference_mode()
def benchmark_model_forward_only(model, cfg: "CFG", batch_size=1, n_warmup=10, n_runs=100):
    """Pure model-forward latency on random tensors -- isolates the
    transformer from face-landmark-detection/preprocessing cost."""
    device = cfg.device
    model.eval()
    frames = torch.randn(batch_size, cfg.num_frames, 3, cfg.img_size, cfg.img_size, device=device)
    lmks_norm = torch.rand(batch_size, cfg.num_frames, cfg.num_landmarks, 3, device=device)
    lmks_px = (torch.rand(batch_size, cfg.num_frames, cfg.num_landmarks, 2, device=device) * cfg.img_size)
    amp_enabled = cfg.amp and device == "cuda"
    amp_dtype = resolve_amp_dtype(cfg)

    def _run_once():
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=amp_enabled):
            model(frames, lmks_norm, lmks_px)
        if device == "cuda":
            torch.cuda.synchronize()
        dt = (time.perf_counter() - t0) * 1000
        return {"preprocess_ms": 0.0, "model_ms": dt, "total_ms": dt}

    for _ in range(n_warmup):
        _run_once()
    records = [_run_once() for _ in range(n_runs)]
    return _summarize_latency(records, tag=f"MODEL-ONLY (batch={batch_size}, random input, no preprocessing)")


# %% CELL 13 -- accuracy evaluation harness (NEW)
# Small-scale correctness check: run the optimized pipeline over a labeled
# manifest and report accuracy / AUC / F1, so you can confirm the SDPA swap
# + index-caching + torch.compile didn't silently change predictions.
# Expect tiny (not zero) numerical drift from SDPA's fused kernels and from
# bf16/fp16 autocast -- that's normal; what matters is the aggregate metrics
# match your original eager-mode numbers within noise.
def evaluate_dataset(model, cfg: "CFG", fa_model, manifest, log_every=10):
    """
    manifest: list of dicts, each {"path": str, "label": 0 or 1, "type": "image"|"video"}
              (label convention: 1 = fake/manipulated, 0 = real -- matches
              the `probs[:, 1]` "p_live"/fake-probability convention used
              elsewhere in this file; flip if your checkpoint uses the
              opposite convention)
    Returns (metrics_dict, probs_list, labels_list) so you can also compute
    ROC curves, confusion matrices, per-sample error analysis, etc. yourself.
    """
    device = cfg.device
    amp_dtype = resolve_amp_dtype(cfg)
    amp_enabled = cfg.amp and device == "cuda"
    model.eval()

    probs_all, labels_all, preds_all, failed = [], [], [], []

    with torch.inference_mode():
        for i, item in enumerate(manifest):
            path, label, kind = item["path"], item["label"], item.get("type", "image")
            try:
                if kind == "video":
                    frames, lmks_norm, lmks_px = _preprocess_video(path, fa_model, cfg)
                else:
                    img_bgr = cv2.imread(path)
                    if img_bgr is None:
                        raise FileNotFoundError(path)
                    frames, lmks_norm, lmks_px = _preprocess_still_image(img_bgr, fa_model, cfg)
            except Exception as e:
                print(f"[skip] {path}: {e}")
                failed.append(path)
                continue

            frames = frames.to(device); lmks_norm = lmks_norm.to(device); lmks_px = lmks_px.to(device)
            with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=amp_enabled):
                logits, *_ = model(frames, lmks_norm, lmks_px)
                p_fake = F.softmax(logits.float(), dim=-1)[:, 1].item()

            probs_all.append(p_fake)
            labels_all.append(int(label))
            preds_all.append(int(p_fake >= THRESHOLD))

            if (i + 1) % log_every == 0:
                print(f"  [{i + 1}/{len(manifest)}] processed")

    labels_arr = np.array(labels_all)
    preds_arr = np.array(preds_all)
    probs_arr = np.array(probs_all)

    metrics = {"n": len(labels_arr), "n_failed": len(failed)}
    if len(labels_arr) == 0:
        print("No samples successfully processed -- check manifest paths.")
        return metrics, probs_all, labels_all

    metrics["accuracy"] = float((labels_arr == preds_arr).mean())
    try:
        from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
        if len(set(labels_all)) > 1:
            metrics["auc"] = float(roc_auc_score(labels_arr, probs_arr))
            metrics["f1"] = float(f1_score(labels_arr, preds_arr))
            metrics["precision"] = float(precision_score(labels_arr, preds_arr, zero_division=0))
            metrics["recall"] = float(recall_score(labels_arr, preds_arr, zero_division=0))
    except ImportError:
        print("[note] sklearn not available -- reporting accuracy only")

    print(f"\n=== Accuracy check: n={metrics['n']} (skipped={metrics['n_failed']}) "
          f"acc={metrics['accuracy']:.4f}"
          + (f"  auc={metrics['auc']:.4f}" if "auc" in metrics else "")
          + (f"  f1={metrics['f1']:.4f}" if "f1" in metrics else "")
          + (f"  prec={metrics['precision']:.4f}" if "precision" in metrics else "")
          + (f"  rec={metrics['recall']:.4f}" if "recall" in metrics else "")
          + " ===")
    return metrics, probs_all, labels_all


def compare_outputs_eager_vs_optimized(cfg: "CFG", ckpt_path: str, fa_model, sample_paths_and_types, atol=1e-2):
    """Numerical sanity check, independent of ground-truth labels: loads TWO
    fresh models from the same checkpoint -- one plain eager (no SDPA-swap
    behavioral difference to worry about since SDPA is mathematically
    equivalent, but this also isolates any torch.compile-specific drift) and
    one with cfg.use_compile as configured -- runs both over the same
    samples, and reports the max absolute difference in fake-probability.
    A few percent drift is expected from bf16/fp16 + fused-kernel changes;
    large drift (e.g. >0.1) means something is actually wrong, not just
    numerical noise, and needs investigating before trusting the optimized
    model's accuracy numbers.
    """
    cfg_eager = cfg.__class__()
    cfg_eager.use_compile = False
    model_eager = load_model_from_checkpoint(cfg_eager, ckpt_path, compile_model=False)
    model_opt = load_model_from_checkpoint(cfg, ckpt_path, compile_model=cfg.use_compile)

    diffs = []
    with torch.inference_mode():
        for path, kind in sample_paths_and_types:
            if kind == "video":
                frames, lmks_norm, lmks_px = _preprocess_video(path, fa_model, cfg)
            else:
                img_bgr = cv2.imread(path)
                frames, lmks_norm, lmks_px = _preprocess_still_image(img_bgr, fa_model, cfg)
            frames_d = frames.to(cfg.device); ln_d = lmks_norm.to(cfg.device); lp_d = lmks_px.to(cfg.device)

            amp_dtype = resolve_amp_dtype(cfg)
            amp_enabled = cfg.amp and cfg.device == "cuda"
            with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=amp_enabled):
                p_eager = F.softmax(model_eager(frames_d, ln_d, lp_d)[0].float(), dim=-1)[:, 1].item()
                p_opt = F.softmax(model_opt(frames_d, ln_d, lp_d)[0].float(), dim=-1)[:, 1].item()

            diff = abs(p_eager - p_opt)
            diffs.append(diff)
            flag = "OK" if diff <= atol else "*** LARGE DIFF ***"
            print(f"  {os.path.basename(path):40s} eager={p_eager:.4f}  optimized={p_opt:.4f}  diff={diff:.4f}  {flag}")

    print(f"\nmax diff = {max(diffs):.4f}, mean diff = {statistics.mean(diffs):.4f} "
          f"(tolerance used for flagging: {atol})")
    return diffs


# %% CELL 14 -- usage
cfg = CFG()
CKPT_PATH = os.path.join(CKPT_INPUT_DIR, "g2v2former_best.pt")   # EDIT: or "g2v2former_lcc_then_celebdf_best.pt", etc.
model = load_model_from_checkpoint(cfg, CKPT_PATH)   # compiled if cfg.use_compile and CUDA available

fa_model = face_alignment.FaceAlignment(
    face_alignment.LandmarksType.THREE_D, flip_input=False, device=cfg.device
)

IMAGE_PATH_FOR_LATENCY = "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_evaluation/spoof/spoof_1000.png"   # EDIT
VIDEO_PATH_FOR_LATENCY = "/kaggle/input/datasets/reubensuju/celeb-df-v2/Celeb-synthesis/id0_id16_0000.mp4"   # EDIT

# NOTE: with torch.compile + mode="reduce-overhead", the first call(s) below
# will be much slower than usual (tracing + CUDA-graph capture). The existing
# n_warmup loops absorb this, same as they already absorbed cudnn autotuning
# -- just don't be alarmed if warmup takes noticeably longer than before.
image_latency = benchmark_image_latency(IMAGE_PATH_FOR_LATENCY, model, cfg, fa_model, n_warmup=2, n_runs=5)
video_latency = benchmark_video_latency(VIDEO_PATH_FOR_LATENCY, model, cfg, fa_model, n_warmup=2, n_runs=5)
model_only_latency = benchmark_model_forward_only(model, cfg, batch_size=1, n_warmup=2, n_runs=5)

# --- small-scale accuracy check on a labeled dataset ---------------------
# EDIT: point this at a small held-out sample (e.g. 50-200 items) of your
# val/test set. `label` convention: 1 = fake, 0 = real (see docstring above).
EVAL_MANIFEST = [
    # {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_training/real/AA5742_id154_s0_112.png", "label": 1, "type": "image"},
    # {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_training/real/AA5742_id154_s0_137.png", "label": 1, "type": "image"},
    # {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_training/real/AA5742_id154_s0_15.png", "label": 1, "type": "image"},
    # {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_training/real/AA5742_id154_s0_47.png", "label": 1, "type": "image"},
    # {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_training/real/AA5742_id154_s0_62.png", "label": 1, "type": "image"},

    # {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_training/spoof/FT720P_G780_REDMI4X_id1_s0_105.png", "label": 0, "type": "image"},
    # {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_training/spoof/FT720P_G780_REDMI4X_id1_s0_120.png", "label": 0, "type": "image"},
    # {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_training/spoof/FT720P_G780_REDMI4X_id1_s0_135.png", "label": 0, "type": "image"},
    # {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_training/spoof/FT720P_G780_REDMI4X_id1_s0_15.png", "label": 0, "type": "image"},
    # {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_training/spoof/FT720P_G780_REDMI4X_id1_s0_150.png", "label": 0, "type": "image"},
    
    {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_evaluation/real/real_0.png", "label": 1, "type": "image"},
    {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_evaluation/real/real_1.png", "label": 1, "type": "image"},
    {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_evaluation/real/real_2.png", "label": 1, "type": "image"},
    {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_evaluation/real/real_3.png", "label": 1, "type": "image"},
    {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_evaluation/real/real_4.png", "label": 1, "type": "image"},
    {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_evaluation/real/real_5.png", "label": 1, "type": "image"},
    {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_evaluation/real/real_6.png", "label": 1, "type": "image"},
    {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_evaluation/real/real_7.png", "label": 1, "type": "image"},
    {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_evaluation/real/real_8.png", "label": 1, "type": "image"},
    {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_evaluation/real/real_9.png", "label": 1, "type": "image"},

    {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_evaluation/spoof/spoof_1001.png", "label": 0, "type": "image"},
    {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_evaluation/spoof/spoof_1002.png", "label": 0, "type": "image"},
    {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_evaluation/spoof/spoof_1003.png", "label": 0, "type": "image"},
    {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_evaluation/spoof/spoof_1004.png", "label": 0, "type": "image"},
    {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_evaluation/spoof/spoof_1005.png", "label": 0, "type": "image"},
    {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_evaluation/spoof/spoof_1006.png", "label": 0, "type": "image"},
    {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_evaluation/spoof/spoof_1007.png", "label": 0, "type": "image"},
    {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_evaluation/spoof/spoof_1008.png", "label": 0, "type": "image"},
    {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_evaluation/spoof/spoof_1009.png", "label": 0, "type": "image"},
    {"path": "/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD/LCC_FASD_evaluation/spoof/spoof_1000.png", "label": 0, "type": "image"},
    
    # {"path": "/kaggle/input/datasets/reubensuju/celeb-df-v2/Celeb-real/id0_0000.mp4", "label": 1, "type": "video"},
    # {"path": "/kaggle/input/datasets/reubensuju/celeb-df-v2/Celeb-real/id0_0002.mp4", "label": 1, "type": "video"},
    # {"path": "/kaggle/input/datasets/reubensuju/celeb-df-v2/Celeb-real/id0_0003.mp4", "label": 1, "type": "video"},
    # {"path": "/kaggle/input/datasets/reubensuju/celeb-df-v2/Celeb-real/id0_0004.mp4", "label": 1, "type": "video"},
    # {"path": "/kaggle/input/datasets/reubensuju/celeb-df-v2/Celeb-real/id0_0001.mp4", "label": 1, "type": "video"},

    # {"path": "/kaggle/input/datasets/reubensuju/celeb-df-v2/Celeb-synthesis/id0_id16_0000.mp4", "label": 0, "type": "video"},
    # {"path": "/kaggle/input/datasets/reubensuju/celeb-df-v2/Celeb-synthesis/id0_id16_0001.mp4", "label": 0, "type": "video"},
    # {"path": "/kaggle/input/datasets/reubensuju/celeb-df-v2/Celeb-synthesis/id0_id16_0002.mp4", "label": 0, "type": "video"},
    # {"path": "/kaggle/input/datasets/reubensuju/celeb-df-v2/Celeb-synthesis/id0_id16_0003.mp4", "label": 0, "type": "video"},
    # {"path": "/kaggle/input/datasets/reubensuju/celeb-df-v2/Celeb-synthesis/id0_id16_0004.mp4", "label": 0, "type": "video"},
]

if EVAL_MANIFEST:
    acc_metrics, probs, labels = evaluate_dataset(model, cfg, fa_model, EVAL_MANIFEST, log_every=10)
else:
    print("[accuracy check] EVAL_MANIFEST is empty -- fill it in with a small "
          "labeled sample to run evaluate_dataset(). Skipping for now.")

# --- optional: eager vs optimized numerical sanity check ------------------
# Confirms the SDPA swap + index-caching + torch.compile aren't silently
# changing predictions, independent of whether you have ground-truth labels.
SANITY_CHECK_SAMPLES = [
    (IMAGE_PATH_FOR_LATENCY, "image"),
    (VIDEO_PATH_FOR_LATENCY, "video"),
]
if SANITY_CHECK_SAMPLES:
    _ = compare_outputs_eager_vs_optimized(cfg, CKPT_PATH, fa_model, SANITY_CHECK_SAMPLES)

face_alignment OK, version: 1.5.0
torch: 2.10.0+cu128 cuda available: True
GPU: Tesla T4 bf16 supported: True
[compile] wrapped model with torch.compile(mode='reduce-overhead') -- first call(s) will be slow while it traces/autotunes, this is expected.


/kaggle/input/datasets/ruwadnaswan/package-df-msib/offline_packages/face_alignment/api.py:130: UserWarning: Compiling face alignment model (one-time cost). Subsequent runs will be faster.
  warnings.warn(
W0901 18:49:20.871000 58 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/variables/functions.py:2056: UserWarning: Dynamo detected a call to a `functools.lru_cache`-wrapped function. Dynamo ignores the cache wrapper and directly traces the wrapped function. Silent incorrectness is only a *potential* risk, not something we have observed. Enable TORCH_LOGS="+dynamo" for a DEBUG stack trace.
  torch._dynamo.utils.warn_once(msg)
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not


=== Latency: IMAGE (spoof_1000.png)  (n=5 runs, batch size 1) ===
  preprocess  mean=  95.60ms  median=  92.83ms  p95=  98.66ms  std=  4.84ms  min=  90.76ms  max= 103.64ms
  model       mean= 126.92ms  median= 127.10ms  p95= 127.20ms  std=  0.79ms  min= 125.82ms  max= 128.12ms
  total       mean= 222.52ms  median= 219.93ms  p95= 224.48ms  std=  5.21ms  min= 217.10ms  max= 231.77ms
  -> throughput: 4.49 inferences/sec (end-to-end, incl. face-landmark detection)

=== Latency: VIDEO (id0_id16_0000.mp4, 8 frames sampled)  (n=5 runs, batch size 1) ===
  preprocess  mean=1140.53ms  median=1139.84ms  p95=1140.03ms  std=  9.88ms  min=1126.98ms  max=1157.77ms
  model       mean= 129.37ms  median= 129.33ms  p95= 129.60ms  std=  0.28ms  min= 128.91ms  max= 129.71ms
  total       mean=1269.90ms  median=1269.33ms  p95=1269.55ms  std=  9.98ms  min=1255.88ms  max=1287.10ms
  -> throughput: 0.79 inferences/sec (end-to-end, incl. face-landmark detection)

=== Latency: MODEL-ONLY (batch=1, random input

In [3]:
!pip install -q fastapi "uvicorn[standard]" pyngrok nest-asyncio

In [4]:
import logging
import threading
import time
import uuid

import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware

nest_asyncio.apply()  # lets uvicorn run inside the Jupyter/Kaggle event loop

logging.basicConfig(
    level="INFO",
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)
logger = logging.getLogger("g2v2former_api")

# Notebook's THRESHOLD (0.1) is tuned as a FAKE-probability cutoff for this
# checkpoint. Override with an env var if you want a different operating point.
API_THRESHOLD = float(os.getenv("THRESHOLD", THRESHOLD))

amp_dtype = resolve_amp_dtype(cfg)

assert model is not None and fa_model is not None, (
    "model / fa_model not found -- run the CELL 14 usage block first"
)

app = FastAPI(title="Face Anti-Spoofing API (G2V2former)", version="2.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],   # tighten to your app's origin once you know it
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/health")
def health():
    return {
        "status": "ok",
        "model_loaded": model is not None,
        "model_path": CKPT_PATH,
        "device": str(cfg.device),
        "threshold": API_THRESHOLD,
    }


@app.post("/api/infer")
async def infer(request: Request):
    if model is None:
        raise HTTPException(status_code=500, detail="Model is not loaded")

    request_id = uuid.uuid4().hex[:8]
    start_time = time.perf_counter()

    content_type = request.headers.get("content-type", "")
    if not content_type.startswith("image/"):
        logger.warning(
            "Inference rejected id=%s reason=invalid_content_type content_type=%s",
            request_id, content_type,
        )
        raise HTTPException(status_code=400, detail="Content-Type must be image/*")

    image_bytes = await request.body()
    if not image_bytes:
        logger.warning("Inference rejected id=%s reason=empty_file", request_id)
        raise HTTPException(status_code=400, detail="Empty file")

    logger.info(
        "Inference started id=%s content_type=%s bytes=%d",
        request_id, content_type, len(image_bytes),
    )

    np_buffer = np.frombuffer(image_bytes, dtype=np.uint8)
    frame_bgr = cv2.imdecode(np_buffer, cv2.IMREAD_COLOR)
    if frame_bgr is None:
        logger.warning("Inference rejected id=%s reason=invalid_image_data", request_id)
        raise HTTPException(status_code=400, detail="Invalid image data")

    try:
        frames, lmks_norm, lmks_px = _preprocess_still_image(frame_bgr, fa_model, cfg)
        p_fake, _, _ = _run_forward(
            model, frames, lmks_norm, lmks_px, cfg, cfg.device, amp_dtype
        )

        prob_spoof = p_fake                 # model convention: softmax index 1 = fake
        prob_real = 1.0 - p_fake
        is_real = prob_spoof > API_THRESHOLD
        confidence = prob_real if is_real else prob_spoof

        inference_time_ms = (time.perf_counter() - start_time) * 1000.0

        logger.info(
            "Inference finished id=%s is_real=%s label=%s confidence=%.4f "
            "prob_real=%.4f prob_spoof=%.4f latency_ms=%.2f",
            request_id, is_real, "REAL_FACE" if is_real else "SPOOF_DETECTED",
            confidence, prob_real, prob_spoof, inference_time_ms,
        )
    except Exception as exc:
        logger.exception("Inference failed id=%s error=%s", request_id, exc)
        raise HTTPException(status_code=500, detail=f"Inference error: {exc}") from exc

    return {
        "request_id": request_id,
        "is_real": is_real,
        "label": "REAL_FACE" if is_real else "SPOOF_DETECTED",
        "confidence": confidence,
        "prob_real": prob_real,
        "prob_spoof": prob_spoof,
        "threshold": API_THRESHOLD,
        "inference_time_ms": round(inference_time_ms, 3),
    }

In [5]:
from pyngrok import ngrok, conf as ngrok_conf

NGROK_AUTH_TOKEN = "3BUBgBtkYCCwdVmvSCokVnp1oWG_85RNGU4z6V9GcGfZ44RMv"  # EDIT: from https://dashboard.ngrok.com/get-started/your-authtoken

def _run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()
time.sleep(2)  # give uvicorn a moment to bind before opening the tunnel

ngrok_conf.get_default().auth_token = NGROK_AUTH_TOKEN
public_tunnel = ngrok.connect(8000, "http")

print(f"Public URL:      {public_tunnel.public_url}")
print(f"Infer endpoint:  {public_tunnel.public_url}/api/infer")
print(f"Health endpoint: {public_tunnel.public_url}/health")

INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
2026-09-01 18:54:02,114 | INFO | pyngrok.ngrok | Opening tunnel named: http-8000-98c04619-0eb6-414a-bd90-a92fd2cd7832


2026-09-01 18:54:02,879 | INFO | pyngrok.process | Overriding default auth token
2026-09-01 18:54:02,901 | INFO | pyngrok.process.ngrok | t=2026-09-01T18:54:02+0000 lvl=info msg="no configuration paths supplied"
2026-09-01 18:54:02,902 | INFO | pyngrok.process.ngrok | t=2026-09-01T18:54:02+0000 lvl=info msg="using configuration at default config path" path=/root/.config/ngrok/ngrok.yml
2026-09-01 18:54:02,903 | INFO | pyngrok.process.ngrok | t=2026-09-01T18:54:02+0000 lvl=info msg="open config file" path=/root/.config/ngrok/ngrok.yml err=<nil>
2026-09-01 18:54:03,046 | INFO | pyngrok.process.ngrok | t=2026-09-01T18:54:03+0000 lvl=info msg="FIPS 140 mode" enabled=false
2026-09-01 18:54:03,053 | INFO | pyngrok.process.ngrok | t=2026-09-01T18:54:03+0000 lvl=info msg="starting web service" obj=web addr=127.0.0.1:4040 allow_hosts=[]
2026-09-01 18:54:03,249 | INFO | pyngrok.process.ngrok | t=2026-09-01T18:54:03+0000 lvl=info msg="client session established" obj=tunnels.session
2026-09-01 18:

Public URL:      https://shila-insuppressible-overnourishingly.ngrok-free.dev
Infer endpoint:  https://shila-insuppressible-overnourishingly.ngrok-free.dev/api/infer
Health endpoint: https://shila-insuppressible-overnourishingly.ngrok-free.dev/health


2026-09-01 18:54:11,758 | INFO | pyngrok.process.ngrok | t=2026-09-01T18:54:11+0000 lvl=info msg="join connections" obj=join id=2e08fc1202c3 l=127.0.0.1:8000 r=114.130.70.58:61397


INFO:     114.130.70.58:0 - "GET /health HTTP/1.1" 200 OK
INFO:     114.130.70.58:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found


2026-09-01 18:56:42,381 | INFO | pyngrok.process.ngrok | t=2026-09-01T18:56:42+0000 lvl=info msg="join connections" obj=join id=c3db35b5f844 l=127.0.0.1:8000 r=114.130.70.56:23664
2026-09-01 18:56:42,537 | INFO | g2v2former_api | Inference started id=ff536afb content_type=image/png bytes=50746
2026-09-01 18:56:45,628 | INFO | g2v2former_api | Inference finished id=ff536afb is_real=True label=REAL_FACE confidence=0.9395 prob_real=0.9395 prob_spoof=0.0605 latency_ms=3245.81


INFO:     114.130.70.56:0 - "POST /api/infer HTTP/1.1" 200 OK


2026-09-01 18:58:48,009 | INFO | pyngrok.process.ngrok | t=2026-09-01T18:58:48+0000 lvl=info msg="join connections" obj=join id=2b4da6ddf45c l=127.0.0.1:8000 r=114.130.70.58:58049
2026-09-01 18:58:48,179 | INFO | g2v2former_api | Inference started id=d75047cc content_type=image/png bytes=117965
2026-09-01 18:58:49,872 | INFO | g2v2former_api | Inference finished id=d75047cc is_real=False label=SPOOF_DETECTED confidence=0.1264 prob_real=0.8736 prob_spoof=0.1264 latency_ms=1862.47


INFO:     114.130.70.58:0 - "POST /api/infer HTTP/1.1" 200 OK


In [ ]:
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    ngrok.disconnect(public_tunnel.public_url)
    print("Tunnel closed.")